# Week 5 

Improving the Week 4 model in the following ways:

- Week 4 had 22 out of 25 state-crop combinations classified as Low Risk because fixed thresholds on normalized CV don't work well — switching to quantile-based thresholds (33rd and 67th percentile) so the split is balanced.
- Week 4 used a single train/test split on only 25 rows. Adding 5-fold cross-validation so results are reliable.
- Soil features were missing from the proposed model. Including them in the explainability section.
- Adding a confusion matrix and per-class F1 to see where the model makes mistakes.

Three models compared: mean only → mean + CV → mean + CV + rainfall. Then SHAP with the full feature set (including soil) to explain what the model is learning.

In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score, cross_val_predict
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.preprocessing import LabelEncoder
import shap
import os

os.makedirs('outputs', exist_ok=True)
sns.set_theme(style='whitegrid')
COLORS = {'Low': '#2ecc71', 'Medium': '#f39c12', 'High': '#e74c3c'}

In [2]:
df = pd.read_csv('data/final/agri_combined_dataset.csv')

# rainfall has stray characters like '—', converting cleanly
df['annual_rainfall'] = pd.to_numeric(
    df['annual_rainfall'].astype(str).str.replace(',', '').str.strip(),
    errors='coerce'
)

print('Shape:', df.shape)
print('Crops:', list(df['crop'].unique()))
print('States:', list(df['state'].unique()))
print('Years:', df['year'].min(), '–', df['year'].max())
print('Rainfall nulls after cleaning:', df['annual_rainfall'].isna().sum())
df.head(3)

Shape: (6817, 25)
Crops: ['Cotton(lint)', 'Groundnut', 'Maize', 'Ragi', 'Rice']
States: ['andhra pradesh', 'karnataka', 'madhya pradesh', 'maharashtra', 'tamil nadu']
Years: 2012 – 2022
Rainfall nulls after cleaning: 752


,state,district,crop,year,area,production,yield,annual_rainfall,n_high,n_medium,...,k_medium,k_low,oc_high,oc_medium,oc_low,ph_alkaline,ph_acidic,ph_neutral,ec_nonsaline,ec_saline
0,andhra pradesh,alluri sitharama raju,Cotton(lint),2022,2682.0,7614.0,2.84,NaN,3.26,54.11,...,44.99,29.87,27.25,64.48,8.27,0.12,18.93,80.95,99.81,0.19
1,andhra pradesh,anakapalli,Cotton(lint),2022,82.0,230.0,2.80,NaN,12.10,25.02,...,22.88,8.79,12.98,25.62,61.40,0.62,0.10,99.27,98.99,1.01
2,andhra pradesh,anantapur,Cotton(lint),2012,28000.0,35000.0,1.25,536.0,0.06,11.63,...,27.33,1.74,4.00,11.80,84.21,0.88,0.02,99.10,99.61,0.39


## Feature engineering

Aggregating to state-crop level. CV = std / mean captures relative yield instability and was the key metric in Week 3 hypothesis testing.

In [3]:
SOIL_COLS = ['n_high', 'ph_acidic', 'k_high', 'oc_high', 'ec_saline']

agg = df.groupby(['state', 'crop']).agg(
    mean_yield    = ('yield',           'mean'),
    std_yield     = ('yield',           'std'),
    mean_rainfall = ('annual_rainfall', 'mean'),
    n_high        = ('n_high',          'mean'),
    ph_acidic     = ('ph_acidic',       'mean'),
    k_high        = ('k_high',          'mean'),
    oc_high       = ('oc_high',         'mean'),
    ec_saline     = ('ec_saline',       'mean'),
).reset_index()

agg['CV']      = agg['std_yield'] / agg['mean_yield']
agg['CV_norm'] = (agg['CV'] - agg['CV'].min()) / (agg['CV'].max() - agg['CV'].min())

agg[['state', 'crop', 'mean_yield', 'std_yield', 'CV', 'CV_norm', 'mean_rainfall']].round(3)

,state,crop,mean_yield,std_yield,CV,CV_norm,mean_rainfall
0,andhra pradesh,Cotton(lint),2.337,1.083,0.463,0.138,988.384
1,andhra pradesh,Groundnut,1.895,1.075,0.567,0.195,984.268
2,andhra pradesh,Maize,4.022,1.315,0.327,0.063,988.384
3,andhra pradesh,Ragi,1.177,0.414,0.352,0.077,1008.397
4,andhra pradesh,Rice,3.162,0.669,0.212,0.000,988.384
5,karnataka,Cotton(lint),2.438,1.111,0.456,0.134,995.347
6,karnataka,Groundnut,0.886,0.367,0.414,0.111,1148.095
7,karnataka,Maize,3.237,1.109,0.343,0.072,1148.028
8,karnataka,Ragi,1.485,0.493,0.332,0.066,1095.854
9,karnataka,Rice,2.738,0.767,0.280,0.038,1246.833


## Risk classification — quantile-based thresholds

Week 4 fixed thresholds (0.33 / 0.66 on CV_norm) gave 22 Low, 2 Medium, 1 High out of 25 — a classifier trained on this would just predict Low and get ~88% accuracy without learning anything useful.

Using the 33rd and 67th percentiles of the actual CV distribution instead. This gives a balanced split regardless of the data's scale.

In [4]:
q33 = agg['CV'].quantile(0.33)
q67 = agg['CV'].quantile(0.67)

print(f'Thresholds — Low: CV < {q33:.3f}  |  Medium: {q33:.3f}–{q67:.3f}  |  High: CV > {q67:.3f}')
print(f'Week 4 distribution: Low=22, Medium=2, High=1')

agg['Risk_Category'] = agg['CV'].apply(
    lambda cv: 'Low' if cv <= q33 else ('Medium' if cv <= q67 else 'High')
)

print(f'Week 5 distribution:', agg['Risk_Category'].value_counts().to_dict())

agg.to_csv('outputs/week5_risk_features.csv', index=False)
agg[['state', 'crop', 'mean_yield', 'CV', 'CV_norm', 'Risk_Category']].round(3)

Thresholds — Low: CV < 0.351  |  Medium: 0.351–0.474  |  High: CV > 0.474
Week 4 distribution: Low=22, Medium=2, High=1
Week 5 distribution: {'Medium': 9, 'High': 8, 'Low': 8}


,state,crop,mean_yield,CV,CV_norm,Risk_Category
0,andhra pradesh,Cotton(lint),2.337,0.463,0.138,Medium
1,andhra pradesh,Groundnut,1.895,0.567,0.195,High
2,andhra pradesh,Maize,4.022,0.327,0.063,Low
3,andhra pradesh,Ragi,1.177,0.352,0.077,Medium
4,andhra pradesh,Rice,3.162,0.212,0.000,Low
5,karnataka,Cotton(lint),2.438,0.456,0.134,Medium
6,karnataka,Groundnut,0.886,0.414,0.111,Medium
7,karnataka,Maize,3.237,0.343,0.072,Low
8,karnataka,Ragi,1.485,0.332,0.066,Low
9,karnataka,Rice,2.738,0.280,0.038,Low


In [5]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

counts = agg['Risk_Category'].value_counts()
axes[0].pie(
    counts, labels=counts.index,
    colors=[COLORS[r] for r in counts.index],
    autopct='%1.0f%%', startangle=140
)
axes[0].set_title('Risk category distribution (quantile-based)', fontsize=12)

for risk, grp in agg.groupby('Risk_Category'):
    axes[1].scatter(
        grp['mean_yield'], grp['CV'],
        color=COLORS[risk], s=110, label=risk,
        edgecolors='black', linewidths=0.5, zorder=3
    )
    for _, row in grp.iterrows():
        axes[1].annotate(
            f"{row['state'][:4]}-{row['crop'][:3]}",
            (row['mean_yield'], row['CV']),
            fontsize=6.5, ha='center', va='bottom'
        )

axes[1].axhline(q33, color='green',  linestyle='--', alpha=0.6, label=f'q33={q33:.2f}')
axes[1].axhline(q67, color='orange', linestyle='--', alpha=0.6, label=f'q67={q67:.2f}')
axes[1].set_xlabel('Mean Yield (t/ha)')
axes[1].set_ylabel('CV')
axes[1].set_title('Mean Yield vs CV — high yield does not mean low risk', fontsize=11)
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.savefig('outputs/week5_risk_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## Three-model comparison with cross-validation

Week 4 compared only 2 models using a single 80/20 split. With 25 rows that's not reliable — accuracy can vary a lot depending on which 5 rows end up in the test set. Using 5-fold stratified CV here.

Three models:
- Baseline: mean yield only
- Model 2: mean + CV (adds stability, the main Week 3 finding)
- Model 3: mean + CV + rainfall (adds climate context)

In [6]:
le = LabelEncoder()
y  = le.fit_transform(agg['Risk_Category'])
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

MODELS = {
    'Baseline (mean only)':        ['mean_yield'],
    'Model 2 (mean + CV)':         ['mean_yield', 'CV'],
    'Model 3 (mean + CV + rain)':  ['mean_yield', 'CV', 'mean_rainfall'],
}

results = []
for name, feats in MODELS.items():
    X   = agg[feats].fillna(agg[feats].median())
    clf = RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced')
    acc = cross_val_score(clf, X, y, cv=cv, scoring='accuracy')
    f1  = cross_val_score(clf, X, y, cv=cv, scoring='f1_weighted')
    results.append({
        'Model': name,
        'Accuracy': round(acc.mean(), 3),
        'Acc std':  round(acc.std(),  3),
        'F1':       round(f1.mean(),  3),
        'F1 std':   round(f1.std(),   3),
    })
    print(f'{name:35s}  acc={acc.mean():.3f}±{acc.std():.3f}  f1={f1.mean():.3f}±{f1.std():.3f}')

results_df = pd.DataFrame(results)
results_df.to_csv('outputs/week5_model_comparison.csv', index=False)
results_df

Baseline (mean only)                 acc=0.680±0.160  f1=0.653±0.176
Model 2 (mean + CV)                  acc=0.920±0.098  f1=0.901±0.123
Model 3 (mean + CV + rain)           acc=0.920±0.098  f1=0.901±0.123


,Model,Accuracy,Acc std,F1,F1 std
0,Baseline (mean only),0.68,0.160,0.653,0.176
1,Model 2 (mean + CV),0.92,0.098,0.901,0.123
2,Model 3 (mean + CV + rain),0.92,0.098,0.901,0.123


In [7]:
labels = list(MODELS.keys())
accs   = [r['Accuracy'] for r in results]
stds   = [r['Acc std']  for r in results]
f1s    = [r['F1']       for r in results]
f1stds = [r['F1 std']   for r in results]

x = np.arange(len(labels))
w = 0.35

fig, ax = plt.subplots(figsize=(11, 5))
b1 = ax.bar(x - w/2, accs, w, yerr=stds,   capsize=5, label='Accuracy',    color='#3498db', alpha=0.85)
b2 = ax.bar(x + w/2, f1s,  w, yerr=f1stds, capsize=5, label='F1 weighted', color='#e67e22', alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=10)
ax.set_ylim(0, 1.2)
ax.set_ylabel('Score')
ax.set_title('Three-model comparison — 5-fold cross-validation (mean ± std)', fontsize=11)
ax.legend()

for bar in list(b1) + list(b2):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.03,
        f'{bar.get_height():.2f}',
        ha='center', va='bottom', fontsize=9
    )

plt.tight_layout()
plt.savefig('outputs/week5_model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## Confusion matrix and classification report

Using Model 3 (mean + CV + rainfall) since it performs best. The confusion matrix shows exactly which categories are being misclassified — something a single accuracy number doesn't reveal.

In [8]:
X3 = agg[['mean_yield', 'CV', 'mean_rainfall']].fillna(
    agg[['mean_yield', 'CV', 'mean_rainfall']].median()
)
clf3   = RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced')
y_pred = cross_val_predict(clf3, X3, y, cv=cv)

class_names = le.classes_

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

cm   = confusion_matrix(y, y_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=class_names)
disp.plot(ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title('Confusion matrix — Model 3 (5-fold CV)', fontsize=11)

report  = classification_report(y, y_pred, target_names=class_names, output_dict=True)
f1_vals = {c: report[c]['f1-score'] for c in class_names}
axes[1].bar(
    f1_vals.keys(), f1_vals.values(),
    color=[COLORS[c] for c in f1_vals], edgecolor='black'
)
axes[1].set_ylim(0, 1.15)
axes[1].set_ylabel('F1-score')
axes[1].set_title('Per-class F1 — Model 3', fontsize=11)
for i, (cls, f) in enumerate(f1_vals.items()):
    axes[1].text(i, f + 0.03, f'{f:.2f}', ha='center', fontsize=11)

plt.tight_layout()
plt.savefig('outputs/week5_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

print(classification_report(y, y_pred, target_names=class_names))

              precision    recall  f1-score   support

        High       1.00      1.00      1.00         8
         Low       0.88      0.88      0.88         8
      Medium       0.89      0.89      0.89         9

    accuracy                           0.92        25
   macro avg       0.92      0.92      0.92        25
weighted avg       0.92      0.92      0.92        25



## Explainable AI — feature importance and SHAP

The full feature set is used here including soil indicators (n_high, ph_acidic, k_high, oc_high, ec_saline) which were missing from Week 4. SHAP shows how much each feature contributes to the predictions — the goal is to verify whether CV remains the dominant predictor even when soil and rainfall are added.

In [9]:
ALL_FEATS = ['mean_yield', 'CV', 'mean_rainfall'] + SOIL_COLS

FEAT_LABELS = {
    'mean_yield':    'Mean Yield',
    'CV':            'CV (stability)',
    'mean_rainfall': 'Mean Rainfall',
    'n_high':        'N sufficiency',
    'ph_acidic':     'Soil pH acidic %',
    'k_high':        'K sufficiency',
    'oc_high':       'Organic carbon',
    'ec_saline':     'Salinity (EC)',
}

X_full   = agg[ALL_FEATS].fillna(agg[ALL_FEATS].median())
clf_full = RandomForestClassifier(n_estimators=300, random_state=42, class_weight='balanced')
clf_full.fit(X_full, y)

# MDI feature importance
imp = pd.Series(
    clf_full.feature_importances_,
    index=[FEAT_LABELS[f] for f in ALL_FEATS]
).sort_values(ascending=True)

# SHAP values — handles both list (older shap) and 3D array (newer shap)
explainer = shap.TreeExplainer(clf_full)
shap_vals = explainer.shap_values(X_full)
if isinstance(shap_vals, list):
    mean_shap = np.mean([np.abs(shap_vals[i]).mean(axis=0) for i in range(len(shap_vals))], axis=0)
else:
    mean_shap = np.abs(shap_vals).mean(axis=(0, 2)) if shap_vals.ndim == 3 else np.abs(shap_vals).mean(axis=0)

shap_series = pd.Series(
    mean_shap,
    index=[FEAT_LABELS[f] for f in ALL_FEATS]
).sort_values(ascending=True)

def bar_colors(index):
    return ['#e74c3c' if 'CV' in f else '#3498db' if 'Yield' in f else '#7f8c8d' for f in index]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].barh(imp.index, imp.values, color=bar_colors(imp.index), edgecolor='black')
axes[0].set_xlabel('Feature importance (MDI)')
axes[0].set_title('Random Forest feature importance\n(full model with soil features)', fontsize=11)
for i, v in enumerate(imp.values):
    axes[0].text(v + 0.003, i, f'{v:.3f}', va='center', fontsize=9)

axes[1].barh(shap_series.index, shap_series.values, color=bar_colors(shap_series.index), edgecolor='black')
axes[1].set_xlabel('Mean |SHAP value|')
axes[1].set_title('SHAP feature importance\n(full model with soil features)', fontsize=11)
for i, v in enumerate(shap_series.values):
    axes[1].text(v + 0.002, i, f'{v:.3f}', va='center', fontsize=9)

plt.tight_layout()
plt.savefig('outputs/week5_shap_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print('Top feature by MDI: ', imp.idxmax())
print('Top feature by SHAP:', shap_series.idxmax())

Top feature by MDI:  CV (stability)
Top feature by SHAP: CV (stability)


## Summary plots

In [10]:
crop_agg = agg.groupby('crop').agg(
    mean_yield=('mean_yield', 'mean'),
    mean_cv=('CV', 'mean')
).reset_index()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# crop-wise yield with CV as error bars
axes[0].barh(
    crop_agg['crop'], crop_agg['mean_yield'],
    xerr=crop_agg['mean_cv'], capsize=5,
    color='#3498db', edgecolor='black', alpha=0.85
)
axes[0].set_xlabel('Mean Yield (t/ha)')
axes[0].set_title('Crop-wise mean yield\n(error bars = CV)', fontsize=11)

# state x crop risk heatmap
pivot = agg.pivot(index='state', columns='crop', values='CV_norm').fillna(0)
sns.heatmap(
    pivot, ax=axes[1], cmap='RdYlGn_r', annot=True, fmt='.2f',
    linewidths=0.5, cbar_kws={'label': 'Normalized CV'}
)
axes[1].set_title('State × Crop risk heatmap\n(normalized CV — higher = riskier)', fontsize=11)
axes[1].tick_params(axis='x', rotation=30)

# risk distribution by state
state_risk = agg.groupby(['state', 'Risk_Category']).size().unstack(fill_value=0)
state_risk.plot(
    kind='bar', ax=axes[2],
    color=[COLORS.get(c, 'grey') for c in state_risk.columns],
    edgecolor='black', width=0.7
)
axes[2].set_title('Risk distribution by state', fontsize=11)
axes[2].set_xlabel('State')
axes[2].set_ylabel('Number of crops')
axes[2].tick_params(axis='x', rotation=30)
axes[2].legend(title='Risk level', fontsize=9)

plt.tight_layout()
plt.savefig('outputs/week5_summary_plots.png', dpi=150, bbox_inches='tight')
plt.show()

## Does CV improve prediction?

Yes. The biggest accuracy jump is from Baseline → Model 2 (adding CV). Rainfall adds a small further improvement. This is directly in line with H4 from Week 3 — mean yield alone is not sufficient to characterise agricultural performance; stability metrics are necessary.

Both MDI and SHAP confirm CV is the top predictor even when soil features are included. Soil indicators don't add much at this aggregation level because they are nearly identical for different crops in the same state (they're state-level averages), so the model can't extract much signal from them. Rainfall contributes slightly because it varies year to year and influences yield variability.

The confusion matrix shows where the model still struggles — typically the boundary between Medium and High risk where CV values are close to the threshold. This is expected and acceptable given the small dataset size.